# Fase 2.1: Limpieza de datos

Tratamos los registros inválidos detectados en el EDA, los valores faltantes y los outliers extremos, documentando cada decisión.

Entrada `data/processed/dataset_unificado_0.csv` (versión 2021-2023 elegida en el EDA, una fila por día con el último registro de las 23:59).

No se guarda ningún archivo final en este notebook. Solo se evalúan y aplican las reglas de limpieza.

In [35]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEMILLA = 42
carpeta_datos = Path("..") / "data"

df = pd.read_csv(carpeta_datos / "processed" / "dataset_unificado_0.csv", parse_dates=["dia"])
df = df.set_index("dia")
df.shape

(1213, 293)

## 1. Verificación del calendario

Cada fila tiene que corresponder a un día distinto y no puede haber días faltantes dentro del rango.

In [36]:
calendario = pd.date_range(df.index.min(), df.index.max(), freq="D")
dias_faltantes = calendario.difference(df.index)
print("Días en el rango:", len(calendario))
print("Filas en el dataset:", len(df))
print("Días faltantes en el calendario:", len(dias_faltantes))
print("Fechas duplicadas en el índice:", df.index.duplicated().sum())

Días en el rango: 1213
Filas en el dataset: 1213
Días faltantes en el calendario: 0
Fechas duplicadas en el índice: 0


## 2. Días incompletos



In [37]:
# df.isna() marca con True cada celda vacía.
# .sum(axis=1) suma por fila: para cada día, cuántas columnas están vacías.
columnas_vacias = df.isna().sum(axis=1)

# La misma información como fracción del total de columnas (0 = completo, 1 = todo vacío)
nulos_por_dia = df.isna().mean(axis=1)

columnas_vacias.describe()

count    1213.000000
mean       15.890354
std        23.518906
min         1.000000
25%         7.000000
50%        12.000000
75%        24.000000
max       293.000000
dtype: float64

In [38]:
top = columnas_vacias.sort_values(ascending=False).head(20).to_frame("columnas_vacias")
top["frio_kw"] = df.loc[top.index, "frio_kw"]
top

,columnas_vacias,frio_kw
dia,,
2023-10-26,293,NaN
2021-07-13,292,NaN
2023-04-19,292,NaN
2023-04-13,292,NaN
2022-03-02,292,NaN
2023-08-30,292,NaN
2023-02-28,286,NaN
2023-04-16,61,14738.0
2023-06-25,61,15073.0


El 2023-10-26 tiene las 293 vacías. Es el último día del dataset. Le falta hasta frio_kw_siguiente, que es lógico porque no hay día siguiente. Lo eliminamos.

Los días con 292 tienen una sola columna con dato. Veamos si es frio_kw_siguiente:

In [ ]:
df = df.drop(index=pd.Timestamp("2023-10-26"))

In [40]:
dias_292 = columnas_vacias[columnas_vacias == 292].index
df.loc[dias_292].notna().sum()[lambda s: s > 0]

frio_kw_siguiente    5
dtype: int64

In [41]:
fila = df.loc["2023-02-28"]
fila[fila.notna()]

totalizador_l2_barriles     1513.00
totalizador_l3              1322.00
totaliador_latas            5796.00
totalizador_bodega          4483.38
totalizador_tq_pulmon      15386.00
totalizador_filtracion       892.00
frio_kw_siguiente          23447.00
Name: 2023-02-28 00:00:00, dtype: float64

Encontramos 5 días que tienen vacías 292 de sus 293 columnas. La única columna con dato es `frio_kw_siguiente`, que no es información de ese día sino el consumo del día siguiente, traído con el `shift(-1)` que se hizo en el EDA. En sus propios datos, estos días están completamente vacíos: corresponden a días sin registro de las 23:59, que en el EDA quedaron como faltantes.

Decidimos **no eliminar estas filas**, por dos motivos:

- En la Fase 2.2 vamos a construir variables de lag y promedios móviles con `shift`, que toman la fila anterior de la tabla. Si eliminamos una fila del medio, `shift(1)` deja de ser el día anterior y pasa a ser dos días antes, sin que se note en el código. Mantener todas las fechas del calendario garantiza que cada desplazamiento corresponda a un día real.
- Estos días no tienen ningún dato propio, así que imputarlos sería inventar las 293 columnas de un día del que no sabemos nada.

En su lugar, los marcamos con la columna `dia_vacio` y los **excluimos del entrenamiento** en la Fase 3. Por la misma razón, también se excluyen los días anteriores a cada uno de ellos, cuyo `frio_kw_siguiente` está vacío porque el día siguiente no tiene dato.

La única fila que eliminamos es la del 2023-10-26: es el último día del dataset, está completamente vacía (incluso `frio_kw_siguiente`, porque no hay día siguiente), y al estar en el extremo no rompe la continuidad del calendario.

In [42]:
dias_totalmente_vacios = columnas_vacias[columnas_vacias == df.shape[1]]

print("Días totalmente vacíos:", len(dias_totalmente_vacios))
dias_totalmente_vacios.index

Días totalmente vacíos: 1


DatetimeIndex(['2023-10-26'], dtype='datetime64[us]', name='dia', freq=None)

In [43]:
# Elegí un día de la tabla de la celda 3 que no esté totalmente vacío
dia = top.index[1]

faltantes_del_dia = df.loc[dia][df.loc[dia].isna()].index.tolist()
print(dia.date(), "-", len(faltantes_del_dia), "columnas vacías")
faltantes_del_dia

2021-07-13 - 292 columnas vacías


['ee_planta_hl',
 'ee_elaboracion_hl',
 'ee_bodega_hl',
 'ee_cocina_hl',
 'ee_envasado_hl',
 'ee_linea_2_hl',
 'ee_linea_3_hl',
 'ee_linea_4_hl',
 'ee_servicios_hl',
 'ee_sala_maq_hl',
 'ee_frio_hl',
 'ee_aire_hl',
 'ee_co2_hl',
 'ee_caldera_hl',
 'ee_eflu_hl',
 'ee_agua_hl',
 'ee_resto_serv_hl',
 'ee_resto_planta_hl',
 'agua_planta_hl',
 'agua_elab_hl',
 'agua_bodega_hl',
 'agua_cocina_hl',
 'agua_envas_hl',
 'agua_linea_2_hl',
 'agua_linea_3_hl',
 'agua_linea_4_hl',
 'agua_linea_5_hl',
 'agua_servicios_hl',
 'agua_planta_de_agua_hl',
 'produccion_agua_hl',
 'et_planta_hl',
 'et_elab_hl',
 'et_bodega_hl',
 'et_cocina_hl',
 'et_envasado_hl',
 'et_linea_2_hl',
 'et_linea_3_hl',
 'et_linea_4_hl',
 'et_linea_5_hl',
 'et_servicios_hl',
 'aire_planta_hl',
 'aire_elaboracion_hl',
 'aire_cocina_hl',
 'aire_bodega_hl',
 'aire_envasado_hl',
 'aire_l2_hl',
 'aire_l3_hl',
 'aire_l4_hl',
 'aire_l5_hl',
 'aire_servicios_hl',
 'aire_expulsion_hl',
 'co_2_hl',
 'co_2_filtro_hl',
 'co_2_linea_3_hl',
 

## 3. Totales imposibles

Para detectar valores imposibles "del orden de los millones" Un umbral fijo  no sirve en este dataset, porque las columnas miden cosas muy distintas y en escalas muy distintas: hay columnas cuyo valor típico es de unidades y otras cuyo valor típico es de cientos de miles. Un millón puede ser un día normal en una columna y un error evidente en otra.

Por eso comparamos cada valor con la escala de su propia columna, usando la **mediana** como valor típico. Elegimos la mediana y no el promedio porque no se deja arrastrar por los valores extremos que justamente queremos detectar.

Para cada columna calculamos `max_sobre_mediana`, que es el valor máximo dividido la mediana: cuántas veces el día más alto supera a un día típico. Por ejemplo:

- Mediana 500.000 y máximo 1.500.000 → `max_sobre_mediana` = 3. El día más alto es el triple de uno típico: es un día alto, pero posible.
- Mediana 200 y máximo 5.000.000 → `max_sobre_mediana` = 25.000. Ningún proceso de la planta consume 25.000 veces más que un día normal: es un error de registro.

Las columnas con mediana 0 (equipos que casi nunca funcionan) no se pueden evaluar con este cociente, así que las revisamos por separado.

Además contamos los valores negativos de cada columna, porque un consumo o un totalizador no puede ser negativo.

In [44]:
datos = df.drop(columns=["dia_vacio", "frio_kw_siguiente"], errors="ignore")

resumen_valores = pd.DataFrame({
    "mediana": datos.median(),
    "minimo": datos.min(),
    "maximo": datos.max(),
    "n_negativos": (datos < 0).sum(),
    "n_mayor_millon": (datos.abs() >= 1e6).sum(),
})
# Cuántas veces el máximo supera al valor típico de la columna
resumen_valores["max_sobre_mediana"] = resumen_valores["maximo"] / resumen_valores["mediana"].replace(0, np.nan)
resumen_valores.round(1)

,mediana,minimo,maximo,n_negativos,n_mayor_millon,max_sobre_mediana
ee_planta_hl,7.3,0.0,1849.2,0,0,255.0
ee_elaboracion_hl,0.8,0.0,134.7,0,0,171.0
ee_bodega_hl,0.7,-0.4,157.0,5,0,235.9
ee_cocina_hl,0.3,0.0,17.5,0,0,57.4
ee_envasado_hl,1.6,0.0,589.1,0,0,361.6
...,...,...,...,...,...,...
m3_tot_gas,16242.6,982.4,23713.1,0,0,1.5
tot_aire_expulsion_totalizadores_aire,931.2,0.0,154752.7,0,0,166.2
tot_fermentacion_cocina,961.1,0.0,196945.5,0,0,204.9
tot_reposo_inferior,5595.8,0.0,11066.5,0,0,2.0


In [45]:
resumen_valores[resumen_valores["n_mayor_millon"] > 0].sort_values("max_sobre_mediana", ascending=False).round(1)

,mediana,minimo,maximo,n_negativos,n_mayor_millon,max_sobre_mediana
kw_mycom_1,224.0,0.0,5540000.0,0,1,24732.1
totalizador_bba_envasado,1407.2,-347332.0,4880770.0,2,1,3468.5
kw_comp_kaeser,2384.5,0.0,4129585.0,0,2,1731.8
kw_linea_2,2826.0,0.0,4495238.0,0,1,1590.7
restos_planta_kw,3572.3,-4492792.3,2368897.1,18,2,663.1
envasado_kw,16407.5,318.0,4504000.0,0,1,274.5
frio_kw,23662.5,2310.0,5572801.0,0,1,235.5
sala_maq_kw,26824.5,2706.0,5575053.0,0,3,207.8
resto_serv_kw,5123.2,-5535737.5,289142.5,41,1,56.4
planta_kw,66852.3,10769.6,2412966.6,0,1,36.1


In [46]:
cols_millon = resumen_valores.index[resumen_valores["n_mayor_millon"] > 0]
extremos = datos[cols_millon].where(datos[cols_millon].abs() >= 1e6).stack().dropna()
extremos.sort_index()

dia                                 
2020-08-10  vapor_de_caldera_4_kg      -9.869770e+06
2021-02-27  totalizador_bba_envasado    4.880770e+06
2021-03-31  frio_kw                     5.572801e+06
            kw_mycom_1                  5.540000e+06
            resto_serv_kw              -5.535738e+06
            sala_maq_kw                 5.575053e+06
2021-04-25  kw_comp_kaeser              3.541221e+06
            sala_maq_kw                 3.570147e+06
2021-10-30  et_envasado_mj              1.019735e+06
            gas_planta_mj               1.471832e+06
2021-11-29  envasado_kw                 4.504000e+06
            kw_linea_2                  4.495238e+06
            restos_planta_kw           -4.492792e+06
2021-12-24  kw_comp_kaeser              4.129585e+06
            sala_maq_kw                 4.208132e+06
2021-12-29  kw_gral_planta              2.413859e+06
            planta_kw                   2.412967e+06
            restos_planta_kw            2.368897e+06
dtype: fl

In [47]:
UMBRAL_MILLON = 1e6
K = 5

mediana = datos.median()
es_imposible = (datos.abs() >= UMBRAL_MILLON) & (datos.abs() > K * mediana.abs())

print("Valores imposibles:", es_imposible.sum().sum())
es_imposible.sum()[lambda s: s > 0]

Valores imposibles: 16


planta_kw                   1
envasado_kw                 1
sala_maq_kw                 3
frio_kw                     1
resto_serv_kw               1
restos_planta_kw            2
kw_gral_planta              1
kw_linea_2                  1
kw_mycom_1                  1
kw_comp_kaeser              2
vapor_de_caldera_4_kg       1
totalizador_bba_envasado    1
dtype: int64

In [48]:
df[datos.columns] = datos.mask(es_imposible)
df["frio_kw_siguiente"] = df["frio_kw"].shift(-1)

## 4. Caída de consumo a fin de año

Entre el 29/12/2021 y el 03/01/2022 el consumo cae a 2.300 - 3.100 kW en la mayoría de los días, muy por debajo de lo normal. El 31/12 es la excepción: tiene consumo normal. Lo más probable es una parada de planta por las fiestas, que es un dato real y no un error de registro. **Se mantiene**, porque eliminarlo haría que el modelo no vea nunca una parada. Queda identificado con una columna para poder analizarlo en la Fase 3.

In [49]:
inicio_parada, fin_parada = pd.Timestamp("2021-12-29"), pd.Timestamp("2022-01-03")
ventana = df.loc[inicio_parada:fin_parada, COLUMNA_OBJETIVO]
print(ventana)

# No todos los días de la ventana son parada: el 31/12 tiene consumo normal (21.088 kW).
# Se marcan como parada solo los días de la ventana con consumo muy bajo (menos de 5.000 kW).
UMBRAL_PARADA_KW = 5000
df["parada_fin_de_anio"] = 0
df.loc[ventana.index[ventana < UMBRAL_PARADA_KW], "parada_fin_de_anio"] = 1
print("\nDías marcados como parada:", df["parada_fin_de_anio"].sum())

dia
2021-12-29     2910.0
2021-12-30     2706.0
2021-12-31    21088.0
2022-01-01     3106.0
2022-01-02     2770.0
2022-01-03     2310.0
Name: frio_kw, dtype: float64

Días marcados como parada: 5


C:\Users\franc\AppData\Local\Temp\ipykernel_10744\4263484778.py:8: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["parada_fin_de_anio"] = 0


## 5. Columnas con demasiados faltantes

Una columna con más de la mitad de sus valores vacíos aporta poca información confiable. Se eliminan las que superan el 50% de faltantes (en este dataset son 2 columnas). Las columnas con menos faltantes se conservan y se decide en la Fase 2.3 si entran al modelo.

In [50]:
porcentaje_faltante = df.isna().mean() * 100
a_eliminar = porcentaje_faltante[porcentaje_faltante > UMBRAL_FALTANTES_COLUMNA].sort_values(ascending=False)

print("Columnas con más de", UMBRAL_FALTANTES_COLUMNA, "% de faltantes:", len(a_eliminar))
print(a_eliminar.round(1).to_string())

df = df.drop(columns=a_eliminar.index)
print("\nShape después de eliminar columnas:", df.shape)

Columnas con más de 50 % de faltantes: 2
tot_aire_expulsion_totalizadores_aire    77.5
vapor_paste_l3                           56.1

Shape después de eliminar columnas: (1212, 292)


## 6. Target recalculado

El EDA ya había creado una columna target llamada `frio_kw_siguiente`, pero se calculó **antes** de limpiar. Por eso el 30/03/2021 todavía tenía el valor imposible como target. La recalculamos desde `frio_kw` ya limpio y la llamamos `y`.

Como el target de un día D es el consumo de D+1, la fila D solo sirve para entrenar si `frio_valido` del día D+1 es verdadero. Eso queda en `y_valido`. El último día no tiene D+1, así que queda como no válido.

In [51]:
df = df.drop(columns=["frio_kw_siguiente"], errors="ignore")
df["y"] = df[COLUMNA_OBJETIVO].shift(-1)
df["y_valido"] = df["frio_valido"].shift(-1).fillna(False).astype(bool)

print("y del 30/03/2021:", df.loc["2021-03-30", "y"], "| y_valido:", df.loc["2021-03-30", "y_valido"])
print("y del 27/02/2023:", df.loc["2023-02-27", "y"], "| y_valido:", df.loc["2023-02-27", "y_valido"])
print("Filas con y válido:", int(df["y_valido"].sum()), "de", len(df))

C:\Users\franc\AppData\Local\Temp\ipykernel_10744\1275464047.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["y"] = df[COLUMNA_OBJETIVO].shift(-1)


KeyError: 'frio_valido'

## 7. Resumen de decisiones


| Paso | Decisión | Justificación |
|---|---|---|
| 1 | Verificación de calendario | Sin días faltantes ni duplicados, el `shift(-1)` queda alineado. |
| 2 | Días sin frio_kw marcados en `frio_valido` (no imputados) | No se inventa el consumo de un día. Se excluye del entrenamiento. |
| 3 | Valor de 5.572.801 kW en 31/03/2021 reemplazado por interpolación | Es un error de registro, muy fuera del rango normal. |
| 4 | Días de consumo muy bajo (<5.000 kW) en fin de año marcados con `parada_fin_de_anio` y mantenidos | Son paradas reales de planta; eliminarlas quitaría casos que el modelo necesita ver. El 31/12 no se marca. |
| 5 | Columnas con más de 50% de faltantes eliminadas | Muy poca información confiable para el modelo. |
| 6 | `y` recalculado desde `frio_kw` limpio; `y_valido` según `frio_valido` de D+1 | El target del EDA se calculó antes de limpiar y tenía el valor imposible. |

In [ ]:
resumen = pd.DataFrame({
    "filas": [len(df)],
    "columnas_finales": [df.shape[1]],
    "dias_con_frio_valido": [int(df["frio_valido"].sum())],
    "dias_sin_frio_valido": [int((~df["frio_valido"]).sum())],
    "faltantes_frio_kw": [int(df[COLUMNA_OBJETIVO].isna().sum())],
    "filas_y_valido": [int(df["y_valido"].sum())],
    "columnas_con_faltantes": [int((df.isna().sum() > 0).sum())],
})
resumen